Import libraries

In [4]:
import time
from codebase_0 import CANN, MLP, cann_training_loop, mlp_train_loop, DeviceLoader, set_seed
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from torch import nn
import torch
import pyro

from gamma_bcann_vi import (
    gamma_build_bcann_guide, GammaBCANN, gamma_fit_glm_mle,
    gamma_bcann_svi_training_loop, GammaCANN, gamma_cann_training_loop,
    gamma_deviance, gamma_evaluate_bcann
)

from gamma_bcann_hmc_nuts import (
    gamma_nuts_diagnostics, gamma_predict_from_samples, gamma_nuts_posterior_stats, gamma_deviance_from_stats, gamma_svi_posterior_stats
)

from bcann_neutra import (
    neutra_diagnostics, neutra_posterior_stats, neutra_deviance_from_stats
)

from codebase_0 import set_seed
from poisson_bcann_vi import PoissonBCANN
from glm_interpretability import (
    glm_mle_with_se, load_vi, vi_glm_draws, hmc_glm_draws,
    vi_f_nn, hmc_f_nn, nn_absorption, absorption_table,
    compare_coefficients, relativity_ratio_table, joint_tail_prob,
    forest_plot, absorption_plot, variance_comparison
)


seed = 42
set_seed(seed)
pyro.clear_param_store()

Load data and perform train/validation/test split

In [2]:
device = torch.device("cpu")

torch.set_default_device("cpu")


dataset = pd.read_csv("freMTPL2sev_transformed.csv", index_col= 0) 
y = torch.tensor(dataset["ClaimAmount"].to_numpy(), dtype=torch.float32)
weights = torch.ones(len(y)) 
names = ["intercept"] + list(dataset.drop(["IDpol", "Exposure","ClaimAmount"], axis = 1).columns)

X = torch.tensor(
    dataset.drop(columns=["IDpol", "Exposure","ClaimAmount"]).to_numpy(), 
    dtype=torch.float32
)   # 48 columns


# Learning-test set split
X_learn, X_test, y_learn, y_test, w_learn, w_test = train_test_split(
    X, y, weights, test_size= 0.1, random_state= seed
)

# Training-CV set split
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X_learn, y_learn, w_learn, test_size= 0.1, random_state= seed
)

learn_scaler = MinMaxScaler().fit(X_learn)
X_learn = learn_scaler.transform(X_learn)
X_test = learn_scaler.transform(X_test)

train_scaler = MinMaxScaler().fit(X_train)
X_train = train_scaler.transform(X_train)
X_val = train_scaler.transform(X_val)

def T(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)

X_train, X_val, X_learn, X_test = map(T, (X_train, X_val, X_learn, X_test))
y_train, y_val, y_learn, y_test = map(T, (y_train, y_val, y_learn, y_test))
w_train, w_val, w_learn, w_test = map(T, (w_train, w_val, w_learn, w_test))

torch.set_default_device(device)
train_loader = DeviceLoader(
        X_train, y_train, w_train,
        batch_size = 2056,
        shuffle= True
)
val_loader = DeviceLoader(
        X_val, y_val, w_val,
        batch_size= 10000
)
learn_loader = DeviceLoader(
        X_learn, y_learn, w_learn,
        batch_size= 2056
)
test_loader = DeviceLoader(
        X_test, y_test, w_test,
        batch_size= 10000
)

In [7]:
from gamma_bcann_vi import SHAPE_SITE, gamma_fit_glm_mle, gamma_posterior_shape
from glm_interpretability import load_vi
from scipy.optimize import minimize_scalar
from scipy.stats import gamma as gamma_dist

def summarise(draws):
    d = torch.as_tensor(draws, dtype=torch.float64).reshape(-1)
    if d.numel() == 1:                                   # point estimate (GLM)
        return {"mean": d.item(), "sd": float("nan"), "2.5%": float("nan"), "97.5%": float("nan")}
    return {"mean": d.mean().item(), "sd": d.std().item(),
            "2.5%": d.quantile(0.025).item(), "97.5%": d.quantile(0.975).item()}

alphas = {}

# GLM: Pearson estimate 1 / phi_hat, and the MLE of alpha with mu held at the GLM fit
glm_w, glm_b, alphas["GLM (Pearson)"] = gamma_fit_glm_mle(X_learn, y_learn, w_learn, random_state=seed)
glm_mu = torch.exp(X_learn @ glm_w.T.squeeze(-1) + glm_b).view(-1).double().numpy()
y_np = y_learn.double().numpy()
nll = lambda log_a: -gamma_dist.logpdf(y_np, a=np.exp(log_a), scale=glm_mu / np.exp(log_a)).sum()
alphas["GLM (MLE)"] = np.exp(minimize_scalar(nll, bounds=(-6, 3), method="bounded").x)

# VI: draws of alpha from the saved guide q
vi_model, vi_guide = load_vi("runs/gamma_bcann_vi.pt", X_learn, y_learn)
alphas["VI"] = gamma_posterior_shape(vi_model, vi_guide, X_learn, num_samples=4000)

# HMC (NUTS) and NeuTra: alpha is a sampled site, stored as [chains, draws]
for name, path in [("HMC (NUTS)", "hmc_results/gamma_hmcnuts_full_set.pt"),
                   ("NeuTra", "hmc_results/gamma_neutra_full_set.pt")]:
    alphas[name] = torch.load(path, map_location="cpu", weights_only=False)["samples"][SHAPE_SITE]

alpha_table = pd.DataFrame({k: summarise(v) for k, v in alphas.items()}).T
alpha_table

,mean,sd,2.5%,97.5%
GLM (Pearson),0.057715,NaN,NaN,NaN
GLM (MLE),0.785501,NaN,NaN,NaN
VI,0.065131,0.003356,0.058935,0.071747
HMC (NUTS),0.784271,0.006175,0.772412,0.796573
NeuTra,0.784323,0.006134,0.772230,0.796225
